# CI/CD with GitHub Actions for MLOps Pipelines

## Objectives

By the end of this lesson, **all learners** will be able to:

* Understand the concept of Continuous Integration (CI) and Continuous Deployment (CD) in MLOps, including differences from traditional software CI/CD.
* Keep track of ML models across experiments using MLflow and automatically log metrics, parameters, and artifacts.
* Store and version models and artifacts in object storage (like S3 or MinIO) with proper versioning and tagging.
* Promote only the best models based on evaluation metrics, ensuring deployments improve model performance.

## Introduction to CI/CD in MLOps

**CI/CD** stands for **Continuous Integration and Continuous Deployment**, two essential concepts in MLOps.

- **Continuous Integration (CI):**  
  Automatically builds and tests code whenever a change is made in the repository. Ensures code changes do not break the pipeline or introduce bugs.  
  *In MLOps, CI often triggers model retraining, testing, and evaluation.*

- **Continuous Deployment (CD):**  
  Automatically deploys ML models or applications once CI checks pass. Allows models to be served or retrained without manual intervention.

### Why CI/CD is Important in MLOps

- **Automation:** Minimizes manual steps in training, testing, and deploying ML models.
- **Consistency:** Ensures reproducible environments using Docker containers, so the same code runs the same way in any environment.
- **Traceability:** All workflow runs are logged, making it easier to track changes, reproduce results, and debug issues.
- **Rapid Iteration:** Allows ML teams to quickly implement changes, retrain models, and update deployments.
- **Collaboration:** Multiple developers can safely contribute, knowing automated checks will catch issues before code is merged.
- **Quality Assurance:** Automated testing ensures scripts, pipelines, and models behave as expected, reducing failures in production.

**Example:**  
Whenever we update `train.py` to tweak hyperparameters or improve preprocessing, the pipeline automatically rebuilds the Docker image, pushes it to a registry, and triggers the Airflow DAG to train and deploy the updated model.

![CI/CD Pipeline Diagram](https://user-images.githubusercontent.com/18515597/142376992-5864e901-621e-4073-85ad-a850cb9a6984.png)

### Explanation of the CI/CD Pipeline Diagram

1. **GitHub Repository**  
   Stores all code, including training scripts, Dockerfiles, Airflow DAGs, and API code. Acts as the single source of truth.

2. **GitHub Actions Workflow**  
   Triggered on code changes (push or pull request). Automates building, testing, and deploying steps.

3. **Docker Images**  
   - **Training image:** Packages ML libraries and training scripts.  
   - **Serving image:** Packages API code to serve the trained model.  
   Ensures consistent environments across development, testing, and production.

4. **Docker Registry (Docker Hub)**  
   Stores the built images so they can be pulled by deployment environments or Airflow.

5. **Airflow DAG**  
   Orchestrates the ML workflow: trains the model, evaluates it, and deploys if performance improves.

6. **MLflow & Object Storage (MinIO/S3)**  
   Tracks experiments, metrics, parameters, and artifacts. Stores trained models for versioning and reproducibility.

### Key Components of GitHub Actions for CI/CD

- **Workflow:** Defined in `.github/workflows/` as YAML files, specifying triggers, jobs, and steps.
- **Job:** A set of steps that run in a specific environment (e.g., Ubuntu runner, Docker container).
- **Step:** Individual commands, such as checking out code, running tests, building Docker images, or triggering Airflow DAGs.
- **Triggers:** Events that start workflows automatically, like `push`, `pull_request`, or scheduled runs.
- **Secrets:** Secure storage for credentials, tokens, and API keys used in workflows.

### Typical CI/CD Workflow in MLOps

1. Developer pushes code or merges a pull request.
2. CI workflow runs automated tests, checks, and validation scripts.
3. Docker images for training and serving are built.
4. Images are pushed to a registry like Docker Hub.
5. Airflow DAGs are triggered to retrain, evaluate, and deploy models.
6. MLflow logs metrics, parameters, and artifacts for tracking and reproducibility.

By implementing CI/CD with GitHub Actions, ML teams can achieve **faster development cycles, reduced errors, and reliable deployment pipelines**.


### Benefits of Using GitHub CI/CD in MLOps

- **Automation:** Removes manual steps for building, testing, and deploying ML pipelines.  
- **Consistency:** Ensures reproducible environments using Docker containers.  
- **Traceability:** Each change is version-controlled and automatically logged in the workflow history.  
- **Collaboration:** Team members can push code safely, knowing automated checks will validate changes.  
- **Rapid Iteration:** Allows ML teams to quickly update models with new data, retrain, and deploy without delays.

### GitHub Actions Overview

- Workflows are defined in YAML files under `.github/workflows/`.  
- Each workflow consists of **jobs**, and each job consists of **steps**.  
- Steps can include running scripts, building Docker images, or triggering other workflows.  
- Triggers include events like `push`, `pull_request`, or scheduled runs.


## Tools We Will Use

1. **GitHub Actions:** Automates workflows directly in GitHub repositories.
2. **Docker:** Containerizes ML training and serving environments for consistency.
3. **Docker Hub (or any registry):** Stores Docker images for deployment.
4. **Airflow:** Orchestrates the ML workflow (training → evaluation → deployment).
5. **MLflow:** Tracks experiments, stores metrics, parameters, and artifacts.
6. **MinIO or S3:** Object storage for persisting model artifacts.

## How CI/CD Works in this MLOps Pipeline

1. **Developer pushes code to GitHub**  
   - Includes: `train.py`, Dockerfiles, Airflow DAGs  
   - Why? GitHub repository acts as a **single source of truth** for all code and workflow configuration.

2. **GitHub Actions workflow is triggered**  
   - Detects changes automatically (push or pull request).  
   - Triggers workflow only on the main branch for production deployment.

3. **Docker images are built**  
   - **Training image:** Includes ML libraries, `train.py`, and MLflow client.  
   - **Serving image:** Includes API code (Flask/Gunicorn) to serve trained models.  
   - Why? Docker ensures consistency across environments (local machine, CI/CD runner, Kubernetes cluster).

4. **Images are pushed to Docker Hub**  
   - Makes images accessible to Airflow, Kubernetes, or other deployment environments.

5. **Airflow DAG is triggered**  
   - Executes the ML pipeline using the newly built images.  
   - Steps: train → evaluate → deploy if model improves.

## Step 1: Organize Your Repository

Your GitHub repository should have a clear structure:

```text
.
├── training-container/
│   ├── Dockerfile     
│   └── train.py         
├── serving-container/
│   └── Dockerfile      
├── dags/
│   └── mlops_dag.py     
└── .github/
    └── workflows/
        └── main.yml     

### Explanation

This repository is structured to **separate training, serving, and workflow automation**, which greatly improves maintainability and clarity.

- **Training container:**  
  Contains all the necessary code and dependencies required for training the ML model. This ensures that the training environment is isolated and reproducible, preventing conflicts with other parts of the project.

- **Serving container:**  
  Packages the API and required libraries to serve trained models. It ensures that the model can be deployed reliably and accessed through a REST API or other endpoints, independent of the training environment.

- **Workflow automation:**  
  Defines how Docker images are built, tested, pushed to a registry, and how the pipeline is triggered. This allows seamless CI/CD integration, ensuring that changes in the code automatically flow through the training, testing, and deployment stages.

### Step 2: Define the GitHub Actions Workflow

Create a workflow file `.github/workflows/main.yml` to automate CI/CD:

```yaml
name: MLOps CI/CD

# Trigger workflow on push to main branch
on:
  push:
    branches:
      - main

jobs:
  build-and-push:
    runs-on: ubuntu-latest

    steps:
      # Step 1: Checkout code
      - uses: actions/checkout@v3

      # Step 2: Login to Docker Hub
      - name: Login to Docker Hub
        uses: docker/login-action@v2
        with:
          username: ${{ secrets.DOCKER_USERNAME }}
          password: ${{ secrets.DOCKER_TOKEN }}

      # Step 3: Build and push training image
      - name: Build and push training image
        uses: docker/build-push-action@v4
        with:
          context: ./training-container
          push: true
          tags: yourrepo/iris-training:v1.0.0-${{ github.sha }}

      # Step 4: Build and push serving image
      - name: Build and push serving image
        uses: docker/build-push-action@v4
        with:
          context: ./serving-container
          push: true
          tags: yourrepo/iris-serving:v1.0.0-${{ github.sha }}

      # Step 5: Trigger Airflow DAG
      - name: Trigger Airflow DAG
        run: |
          curl -X POST "http://<AIRFLOW_HOST>:8080/api/v1/dags/mlops_pipeline/dagRuns" \
          -H "Content-Type: application/json" \
          -d '{"conf": {}}' \
          -u ${{ secrets.AIRFLOW_USER }}:${{ secrets.AIRFLOW_PASSWORD }}

### Explanation of Each Step

- **Checkout:**  
  Downloads the repository to the runner machine. Without this step, the workflow would have no access to the code or configuration files.

- **Docker Login:**  
  Authenticates to Docker Hub using stored secrets. This ensures that Docker images can be securely pushed to the registry without exposing credentials.

- **Build Training Image:**  
  Packages `train.py` and all required dependencies into a container. This allows the training environment to be **consistent, reproducible, and portable** across different machines or clusters.

- **Build Serving Image:**  
  Packages API code into a separate container for serving. This ensures that the deployed model can run reliably in production and respond to client requests.

- **Trigger Airflow DAG:**  
  Automatically starts the ML workflow. The DAG will train, evaluate, and deploy models, ensuring that the latest code changes flow through the pipeline with minimal manual intervention.

### Step 3: Testing in Workflow

It is recommended to test the training container before pushing it to production:

```yaml
- name: Run tests
  run: docker run yourrepo/iris-training:v1.0.0-${{ github.sha }} pytest

### Why Testing is Important

- Ensures that the training script runs correctly inside the container.
- Detects missing dependencies or errors before deployment.
- Helps maintain pipeline stability and reliability.

### Step 4: Best Practices

- Always use **secrets** for credentials to avoid exposing sensitive information.
- Keep Dockerfiles **simple and focused**; separate training and serving containers.
- **Version images** using timestamps or commit hashes for traceability.
- **Test before push** to detect issues early and maintain stable builds.
- Keep workflows readable; consider splitting into multiple jobs for clarity.


### Step 5: Summary

- CI/CD automates building, testing, and deploying ML pipelines.
- GitHub Actions ensures the pipeline reacts automatically to code changes.
- Docker containers guarantee consistent environments for both training and serving.
- Airflow orchestrates the ML pipeline, allowing automated retraining and deployment.
- MLflow combined with MinIO/S3 ensures experiments, models, and artifacts are tracked and persisted.

## Key Insights

- **Automated Pipelines Reduce Errors:**  
  CI/CD ensures that every change in code automatically goes through testing, building, and deployment, minimizing human errors.

- **Consistent Environments:**  
  Docker containers guarantee that training and serving environments are identical across local, staging, and production setups.

- **Efficient Model Management:**  
  MLflow with object storage (MinIO/S3) allows versioning of models and artifacts, ensuring reproducibility and easy rollback if needed.

- **Orchestration with Airflow:**  
  Automates the full ML workflow (training → evaluation → deployment), making retraining and updating models seamless.

- **Best Practices Improve Stability:**  
  Using secrets, testing containers, versioning images, and separating training/serving containers ensures a reliable, maintainable, and secure pipeline.

- **End-to-End Automation:**  
  The combination of GitHub Actions, Docker, Airflow, and MLflow provides a fully automated MLOps pipeline that reduces manual intervention while maintaining consistency.